# Exercice 05 – Corrigé

Serveur de départ (Qwen3-8B, 4 slots de 12 288 tokens) :

```bash
llama-server -m Qwen3-8B-Q4_K_M.gguf -c 49152 -np 4 -fa on -ngl all --reasoning off --port 8080
```

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor
from datetime import date
from pathlib import Path
from typing import Literal

import polars as pl
import pymupdf
from openai import OpenAI
from pydantic import BaseModel, Field

client = OpenAI(base_url="http://127.0.0.1:8080/v1", api_key="x", max_retries=0, timeout=900)


class Decision(BaseModel):
    point: int
    intitule: str
    statut: Literal["approuvé", "rejeté", "reporté", "retiré", "information"]
    votes_pour: int | None
    votes_contre: int | None
    abstentions: int | None


class PV(BaseModel):
    resume: str = Field(description="Résumé de la séance en 150 à 250 mots")
    decisions: list[Decision]
    prochaine_reunion: date | None


SYSTEM = """Tu résumes des retranscriptions de conseils d'administration et tu en extrais les décisions.
Une décision par point de l'ordre du jour, y compris les points pour information et les points retirés.
« À l'unanimité » : tous les administrateurs qui votent sont pour ; tiens compte de ceux qui ont quitté la séance.
Si une décision est reportée plus tard dans la séance, c'est le statut final qui compte."""

## 1. La fonction

In [ ]:
def resumer(chemin: str) -> dict:
    texte = "\n".join(page.get_text() for page in pymupdf.open(chemin))

    debut = time.perf_counter()
    r = client.chat.completions.create(
        model="local",
        messages=[{"role": "system", "content": SYSTEM},
                  {"role": "user", "content": f"<document>\n{texte}\n</document>"}],
        response_format={"type": "json_schema",
                         "json_schema": {"name": "pv", "schema": PV.model_json_schema()}},
        temperature=0,
        max_tokens=2500,
    )
    duree = time.perf_counter() - debut

    t = r.model_extra["timings"]
    return {
        "fichier": Path(chemin).name,
        "duree_s": round(duree, 1),
        "prompt_tokens": r.usage.prompt_tokens,
        "prompt_traite": t["prompt_n"],          # hors partie réutilisée depuis le cache (system prompt)
        "prompt_ms": round(t["prompt_ms"]),
        "tokens_generes": t["predicted_n"],
        "tok_s": round(t["predicted_per_second"], 1),
        "finish_reason": r.choices[0].finish_reason,
        "pv": PV.model_validate_json(r.choices[0].message.content),
    }


res = resumer("pv_ca/pv_01.pdf")
print({k: v for k, v in res.items() if k != "pv"})
res["pv"].decisions[:3]

## 2. Les 20 PV avec 4 clients

In [ ]:
def traiter(n_clients: int) -> tuple[pl.DataFrame, float]:
    fichiers = sorted(str(f) for f in Path("pv_ca").glob("*.pdf"))
    debut = time.perf_counter()
    with ThreadPoolExecutor(max_workers=n_clients) as ex:
        resultats = list(ex.map(resumer, fichiers))
    total = time.perf_counter() - debut

    print(f"{n_clients} clients : {len(fichiers)} PV en {total:.0f} s → {len(fichiers) / total * 60:.1f} PV/minute")
    return pl.DataFrame([{k: v for k, v in r.items() if k != "pv"} for r in resultats]), total


df, total = traiter(4)
df

## 3. Comparer des configurations

Pour chaque configuration : arrêter le serveur, le relancer avec les nouvelles options, puis exécuter la deuxième cellule ci-dessous en changeant l'étiquette. La VRAM se lit avec `nvidia-smi` dans un terminal.

| Étiquette | Options serveur | Clients |
|---|---|---|
| `np1` | `-c 12288 -np 1` | 4 |
| `np4` | `-c 49152 -np 4` | 4 |
| `np8-q8` | `-c 98304 -np 8 -ctk q8_0 -ctv q8_0` | 8 |
| `np4-ub2048` | `-c 49152 -np 4 -ub 2048` | 4 |
| `np4-ngram` | `-c 49152 -np 4 --spec-type ngram-mod` | 4 |

In [ ]:
comparaison = []   # à exécuter une seule fois

In [ ]:
ETIQUETTE, N_CLIENTS = "np4", 4        # à changer à chaque configuration

d, total = traiter(N_CLIENTS)
comparaison.append({
    "config": ETIQUETTE,
    "clients": N_CLIENTS,
    "duree_totale_s": round(total),
    "pv_par_min": round(len(d) / total * 60, 1),
    "duree_moy_par_pv_s": round(d["duree_s"].mean(), 1),
    "tok_s_moyen": round(d["tok_s"].mean(), 1),
})
pl.DataFrame(comparaison)

## Ce qu'on s'attend à observer

À confirmer avec vos mesures.

- **`np1` avec 4 clients** : les requêtes passent une par une, la durée totale est celle d'un traitement séquentiel. La durée par PV mesurée côté client augmente, parce qu'elle inclut l'attente dans la file du serveur.
- **`np4`** : les PV/minute augmentent nettement, mais `tok_s_moyen` baisse. Chaque réponse est plus lente, et pourtant l'ensemble va plus vite, parce que les générations simultanées partagent la lecture des poids en mémoire.
- **`np8-q8`** : quantiser le cache permet de faire tenir 8 slots confortablement. Le gain par rapport à `np4` est plus faible, le GPU approchant de sa limite de calcul.
- **`-ub 2048`** : accélère surtout le traitement du prompt (`prompt_ms`). L'effet sur la durée totale reste limité, parce que c'est la génération du JSON qui domine.
- **`ngram-mod`** : peut accélérer la génération quand la réponse reprend des passages du texte. Le gain est modeste sur un résumé structuré.